# Tech Note - Sine Sweep

SinSweepControllerクラスは、操作量としてサインスイープ信号を出力する。<br>
その理論式をこの技術ノートに記す。

# 1. 経時変化する周波数 (周波数は経過時間$t$の関数)

制御システムの周波数応答を調べるため、周波数を変えながら正弦波状の指令値や操作量を出力する場合を考える。<br>
周波数を式(1)によって変化させる。

* $t$ [s]：経過時間
* $f(t)$ [Hz]：経過時間$t$での周波数
* $f_{start}$ [Hz]：掃引開始周波数
* $f_{end}$ [Hz]：掃引終了周波数
* $T$ [s]：スイープ時間（周波数を変化させる時間）

$$
f(t) = f_{start} \cdot \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} \quad \quad (1)
$$

幅広い範囲の周波数を効率良くテストするため、周波数は足し算(1Hz --> 2Hz --> 3Hz)ではなく、<br>
掛け算(1Hz --> 10Hz --> 100Hz)で変化させる。

# 2. 正弦波 (一定周波数)

基本として、一定周波数の正弦波信号の式を確認しておく。

* $x(t)$ [-]：経過時間$t$での信号
* $A$ [-]：振幅
* $f$ [Hz]：周波数 (一定)
* $\omega$ [rad/s]：角速度
* $\theta (t)$ [rad]：位相 (経時変化)

$$
x(t) = A \sin (\theta(t)) = A \sin \left( \omega t \right) = A \sin \left( 2 \pi f t \right) \quad \quad (2)
$$

# 3. 正弦波 (経時変化周波数)

## 3.1 経時変化周波数の分解

経時変化周波数の式(1)を分解して考える。

式(1)から経過時間$t$を除外すると、周波数の変化率となる。<br>
変化倍率を指数関数的に変えるため、以降、これを "指数変化率" と呼ぶことにする。
$$
fの指数変化率 = f_{start} \cdot \left( \frac{f_{end}}{f_{start}} \right)^{\frac{1}{T}} \quad \quad (3)
$$

この指数変化率に経過時間$t$を掛けることで、$t$における周波数が確定する。
$$
f(t) = f_{start} \cdot \left( \frac{f_{end}}{f_{start}} \right)^{\frac{1}{T} \times t} \quad \quad (4)
$$

このように、周波数 = 指数変化率 × 経過時間 と分解できる。

## 3.2 速度と位置のアナロジー

### 3.2.1 経時変化速度の分解

これは、経時変化する速度 $v(t)$ と同様である。
<br>速度$V$まで時間$T$をかけて加速する場合を考える。以下のように変数を定義する。
* $V$ [m/s]：目標速度
* $T$ [s]：加速時間
* $v(t)$ [m/s]：経過時間$t$での速度
* $p(t)$ [m]：経過時間$t$での位置

目標速度$V$と加速時間$T$の比は速度の変化率(加速度)である。
$$
速度vの変化率 = \frac{V}{T} \quad \quad (5)
$$

この変化率に経過時間$t$を掛けることで、$t$における速度が確定する。
$$
v(t) = \frac{V}{T} \times t \quad \quad (6)
$$

このように、速度 = 変化率 × 経過時間 と分解できる。

### 3.2.2 経時変化速度による位置変化量

一定速度の場合は 位置変化量 = 速度 × 時間 だが、
時々刻々と速度が変わる場合、この関係は成り立たない。

ペースを変えながら変わる位置の変化量を求めるには、積分が必要。

$$
p(t) = \int_0^t v(t) dt = \int_0^t \frac{V}{T} \times t dt = \left[ \frac{V}{2T} t^2 \right]_0^t = \frac{V}{2T} t^2 \quad \quad (7)
$$

## 3.3 経時変化周波数の位相

同様に、一定周波数(=一定角速度)の場合は 位相変化量 = 角速度 × 時間だが、時々刻々と周波数が変わる場合、この関係は成り立たない。

ペースを変えながら変わる位相の変化量を求めるには、積分が必要。

* $f(t)$ [Hz]：周波数 (経時変化)
* $\omega(t)$ [rad/s]：角速度 (経時変化)
* $\theta (t)$ [rad]：位相 (経時変化)

$$
\begin{align}
\theta(t) &= \int_0^t \omega(t) dt = \int_0^t 2 \pi \cdot  f(t) dt \notag \\
&= \int_0^t 2 \pi \cdot  f_{start} \cdot \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} dt \notag \\
&= 2 \pi \cdot f_{start} \int_0^t \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} dt \notag \\
&= 2 \pi \cdot f_{start} \left[ \frac{1}{\ln \left( \frac{f_{end}}{f_{start}} \right)} \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}}  \cdot T \right]_0^t \notag \\
&= \frac{2 \pi \cdot  f_{start} \cdot  T}{\ln \left( \frac{f_{end}}{f_{start}} \right)} \left[\left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} \right]_0^t \notag \\
&= \frac{2 \pi \cdot f_{start} \cdot  T}{\ln \left( \frac{f_{end}}{f_{start}} \right)} \left( \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} -1 \right) \notag \quad \quad (8)

\end{align}
$$

このように、経時変化する周波数の下では、位相は指数関数と対数関数の組み合わせで変化する。

## 3.4 経時変化周波数の正弦波形

式(8)の位相を正弦波の式に適用し、経時変化周波数の正弦波信号の式を得る。

$$
\begin{align}
x(t) &= A \sin \left( \theta(t) \right) \notag \\
&= A \sin \left( \frac{2 \pi \cdot f_{start} \cdot  T}{\ln \left( \frac{f_{end}}{f_{start}} \right)} \left( \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} -1 \right) \right) \notag \quad \quad (9)
\end{align}
$$

$t = T$ の場合、

$$
x(t) = A \sin \left( \frac{2 \pi \cdot  T \cdot \left(f_{end} - f_{start} \right)}{\ln \left( \frac{f_{end}}{f_{start}} \right)} \right)  \quad \quad (10)
$$


## 3.5 SinSweepControllerクラス 実装理論式

SinSweepControllerクラスの実装用に項を分けた式を記す。

$$
log_{term} = \ln \left( \frac{f_{end}}{f_{start}} \right) \quad \quad (11)
$$

$$
term_1 = \frac{2 \pi \cdot f_{start} \cdot  T}{log_{term}} \quad \quad (12)
$$

$$
term_2 = \left( \left( \frac{f_{end}}{f_{start}} \right)^{\frac{t}{T}} -1 \right) \quad \quad (13)
$$

$$
phase = term_1 \times term_2 \quad \quad (14)
$$

$$
force = A \times \sin(phase) \quad \quad (15)
$$